In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import sqlite3


# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [6]:
DB_PATH = "db.sqlite3"
OUT_CSV = "data/joined_data.csv"      # <-- куди зберегти

# 1) Підключення
conn = sqlite3.connect(DB_PATH)

# 2) JOIN трьох таблиць (типовий варіант)
query = """
SELECT
    o.id            AS order_id,
    o.created_at    AS order_created_at,
    o.status        AS order_status,

    oi.id           AS order_item_id,
    oi.product_id   AS product_id,
    oi.quantity     AS quantity,
    oi.price        AS item_price,     -- якщо є
    oi.total        AS item_total,     -- якщо є

    p.name          AS product_name,
    p.category      AS product_category,
    p.price         AS product_price   -- якщо є
FROM orders o
JOIN order_items oi ON oi.order_id = o.id
JOIN products p     ON p.id = oi.product_id
;
"""

# 3) Читаємо в pandas напряму з SQL
df = pd.read_sql_query(query, conn)

# 4) Закриваємо конект
conn.close()

# 5) Зберігаємо в CSV
df.to_csv(OUT_CSV, index=False)

# 6) Контрольна перевірка
print("Rows:", len(df))
print(df.head())


OperationalError: unable to open database file

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_products = (
    df.groupby("product_name", as_index=False)["quantity"]
      .sum()
      .sort_values("quantity", ascending=False)
      .head(10)
)

plt.figure(figsize=(9, 9))

labels = top_products["product_name"]
sizes = top_products["quantity"].values
total = sizes.sum()

plt.pie(
    sizes,
    labels=labels,
    autopct=lambda pct: f"{pct:.1f}% ({int(round(pct/100 * total))})",
    startangle=140
)

plt.title("Top 10 products by quantity (share of total in Top 10)")
plt.axis("equal")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [7]:
# 1) Визначаємо, з якої колонки брати ціну продукту
# (під твоє "joined df" з Task 0)
price_col = None
for cand in ["product_price", "price", "item_price"]:
    if cand in df.columns:
        price_col = cand
        break

if price_col is None:
    raise KeyError("Can't find price column. Expected one of: product_price, price, item_price")

# 2) Рахуємо Item Price (виручка по позиції замовлення)
df = df.copy()  # щоб не ловити SettingWithCopyWarning
df["item_price"] = df[price_col].astype(float) * df["quantity"].astype(float)

# 3) Top-10 продуктів за сумарною виручкою
top_revenue = (
    df.groupby("product_name", as_index=False)["item_price"]
      .sum()
      .sort_values("item_price", ascending=False)
      .head(10)
)

# 4) Відсотки
top_revenue["percentage"] = top_revenue["item_price"] / top_revenue["item_price"].sum() * 100

plt.figure(figsize=(9, 9))

labels = top_revenue["product_name"]
sizes = top_revenue["item_price"].values
total = sizes.sum()

plt.pie(
    sizes,
    labels=labels,
    autopct=lambda pct: f"{pct:.1f}% ({pct/100 * total:,.0f})",
    startangle=140
)

plt.title("Top 10 products by revenue (share of total revenue in Top 10)")
plt.axis("equal")
plt.show()

NameError: name 'df' is not defined

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
date_col = None
for cand in ["order_created_at", "created_at", "order_datetime", "datetime"]:
    if cand in df.columns:
        date_col = cand
        break

if date_col is None:
    raise KeyError("Date column not found. Expected order_created_at / created_at / order_datetime")

# конвертуємо у datetime та витягуємо годину
df[date_col] = pd.to_datetime(df[date_col])
df["order_hour"] = df[date_col].dt.hour

df[["order_hour"]].head()

revenue_by_hour = (
    df.groupby("order_hour", as_index=False)["item_price"]
      .sum()
      .sort_values("order_hour")
)

all_hours = pd.DataFrame({"order_hour": range(24)})

revenue_by_hour = all_hours.merge(
    revenue_by_hour,
    on="order_hour",
    how="left"
).fillna(0)

plt.figure(figsize=(12, 6))

plt.bar(
    revenue_by_hour["order_hour"],
    revenue_by_hour["item_price"]
)

plt.xticks(range(24))
plt.xlabel("Order Hour (0–23)")
plt.ylabel("Total Revenue")
plt.title("Total Restaurant Revenue by Order Hour")

plt.show()


# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [8]:
date_col = None
for cand in ["order_created_at", "created_at", "order_datetime", "datetime"]:
    if cand in df.columns:
        date_col = cand
        break

if date_col is None:
    raise KeyError("Date column not found. Expected order_created_at / created_at / order_datetime")

# 1) datetime + день тижня
df[date_col] = pd.to_datetime(df[date_col])

# Monday=0 ... Sunday=6
df["order_dow_num"] = df[date_col].dt.dayofweek
df["order_dow"] = df[date_col].dt.day_name()

# 2) revenue by day-of-week (зберігаємо порядок Mon..Sun)
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

revenue_by_dow = (
    df.groupby("order_dow", as_index=False)["item_price"]
      .sum()
)

revenue_by_dow["order_dow"] = pd.Categorical(revenue_by_dow["order_dow"], categories=dow_order, ordered=True)
revenue_by_dow = revenue_by_dow.sort_values("order_dow").reset_index(drop=True)

plt.figure(figsize=(12, 6))
plt.bar(revenue_by_dow["order_dow"], revenue_by_dow["item_price"])
plt.xlabel("Order Day of Week")
plt.ylabel("Total Revenue")
plt.title("Total Restaurant Revenue by Day of Week")
plt.xticks(rotation=30)
plt.show()

best_day = revenue_by_dow.loc[revenue_by_dow["item_price"].idxmax(), "order_dow"]
worst_day = revenue_by_dow.loc[revenue_by_dow["item_price"].idxmin(), "order_dow"]

print("Best revenue day:", best_day)
print("Lowest revenue day:", worst_day)

NameError: name 'df' is not defined